# pilot_paper 结果闭合入口

该 Notebook 用于在 Colab 中完成 pilot_paper 结果闭合: 挂载 Google Drive、拉取当前仓库、从 ``SLM_WM_DRIVE_RESULT_ROOT`` 物化前序结果包、重建 attack matrix、external baseline formal import、internal ablation、pilot_paper fixed-FPR 共同协议记录, 并把完整结果包写回 Google Drive。

Notebook 只调度 `scripts/` 中的 repository commands, 不直接手写正式 records、tables、figures 或 reports。


## 运行前准备

1. 确认前序方法主流程和 baseline Notebook 已把结果包写入 ``SLM_WM_DRIVE_RESULT_ROOT`/`。
2. 该入口不需要 GPU, 但需要能够访问 Google Drive。
3. 如果本次需要干净复盘, 应先清理 ``SLM_WM_DRIVE_RESULT_ROOT`/` 后重新运行前序 Notebook, 再运行本入口。
4. 输出完整结果包默认写入 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `complete_result_package``。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')
if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)
subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="paper_result_closure",
)
print(paper_run_environment)


In [ ]:
from pathlib import Path

package_search_root = Path(os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'])
assert package_search_root.exists(), f'未找到当前论文运行层级的 Google Drive 结果目录: {package_search_root}'
zip_count_by_dir = {
    child.name: len(list(child.glob('*.zip')))
    for child in sorted(package_search_root.iterdir())
    if child.is_dir()
}
print({'package_search_root': str(package_search_root), 'zip_count_by_dir': zip_count_by_dir})


In [ ]:
import os

from paper_workflow.colab_utils.paper_result_closure import run_paper_result_closure_commands

paper_result_closure_summary = run_paper_result_closure_commands(
    package_search_root=os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'],
    complete_drive_output_dir=os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR'],
    target_fpr=os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'],
    paper_run_name=os.environ['SLM_WM_PAPER_RUN_NAME'],
)
paper_result_closure_summary


In [ ]:
paper_result_closure_summary
